In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

print("All libraries imported successfully!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import io

filename = next(iter(uploaded))

xls = pd.ExcelFile(io.BytesIO(uploaded[filename]))
print("Available sheets:", xls.sheet_names)

In [ ]:
# Load the Regression 2.1 dataset
df = pd.read_excel(xls, sheet_name="Regression_2.1")

# Check dataset dimensions
print("Dataset shape:", df.shape)
print("Number of matches:", len(df))

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Display first 5 matches
display(df.head())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Define our 8 explanatory variables
features = [
    'Team_A_FIFA_Rank',
    'Team_B_FIFA_Rank',
    'Team_A_Form_Points',
    'Team_B_Form_Points',
    'Team_A_Avg_GF',
    'Team_B_Avg_GF',
    'Team_A_Avg_GA',
    'Team_B_Avg_GA'
]

# Define dependent variable
target = 'Goal_Difference'

# Check missing values
print("Missing values:")
print(df[features + [target]].isnull().sum())

# Generate descriptive statistics
print("\nDescriptive Statistics:")
display(df[features + [target]].describe().round(2))

In [ ]:
# Calculate correlations
correlation_matrix = df[features + [target]].corr()

# Create correlation heatmap
plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title("Correlation Heatmap - FIFA World Cup 2026")
plt.tight_layout()
plt.show()

In [ ]:
# Define explanatory variables and target
X = df[features]
y = df[target]

# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Create and train the multiple linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict goal difference for test matches
y_pred = model.predict(X_test)

# Evaluate model performance
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Training matches:", len(X_train))
print("Testing matches:", len(X_test))

print("\nModel Performance:")
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R-squared:", round(r2, 3))

# Display regression coefficients
coefficients = pd.DataFrame({
    "Variable": features,
    "Coefficient": model.coef_
})

print("\nRegression Coefficients:")
display(coefficients.round(4))

print("Intercept:", round(model.intercept_, 4))

In [ ]:
# OLS regression analysis
X_train_ols = sm.add_constant(X_train)

ols_model = sm.OLS(y_train, X_train_ols).fit()

print(ols_model.summary())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
# Calculate VIF with an intercept
X_vif = sm.add_constant(X_train)

vif_data = pd.DataFrame({
    "Variable": features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values, i
        )
        for i in range(1, X_vif.shape[1])
    ]
})

display(vif_data.round(2))

In [ ]:
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan

# Training residuals from OLS
fitted_values = ols_model.fittedvalues
residuals = ols_model.resid

# 1. Residuals vs fitted values
plt.figure(figsize=(8, 5))
sns.scatterplot(x=fitted_values, y=residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Fitted Goal Difference")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.show()

# 2. Q-Q plot
sm.qqplot(residuals, line='45', fit=True)
plt.title("Q-Q Plot of Residuals")
plt.show()

# 3. Residual distribution
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title("Distribution of Regression Residuals")
plt.xlabel("Residual")
plt.show()

# 4. Breusch-Pagan test for heteroscedasticity
bp = het_breuschpagan(residuals, X_train_ols)

print("Breusch-Pagan Test")
print("LM statistic:", round(bp[0], 4))
print("LM p-value:", round(bp[1], 4))
print("F statistic:", round(bp[2], 4))
print("F p-value:", round(bp[3], 4))

## Date-grouped chronological cross-validation
This final evaluation splits unique match dates, keeping every match on a given date together. It compares multiple linear regression with a mean-prediction baseline over three chronological folds. The historical input data still requires independent verification.


In [ ]:
# Date-grouped chronological validation: no calendar date appears in both sets
from sklearn.model_selection import TimeSeriesSplit
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

ordered = df.copy()
ordered['Date'] = pd.to_datetime(ordered['Date'])
ordered = ordered.sort_values(['Date', 'Match_ID']).reset_index(drop=True)
unique_dates = np.array(sorted(ordered['Date'].unique()))
cv = TimeSeriesSplit(n_splits=3)
rows = []

for fold, (tr_dates_idx, te_dates_idx) in enumerate(cv.split(unique_dates), 1):
    train_dates = unique_dates[tr_dates_idx]
    test_dates = unique_dates[te_dates_idx]
    train = ordered[ordered['Date'].isin(train_dates)]
    test = ordered[ordered['Date'].isin(test_dates)]
    assert train['Date'].max() < test['Date'].min(), 'Date overlap detected'
    for label, estimator in [
        ('Linear Regression', LinearRegression()),
        ('Mean Baseline', DummyRegressor(strategy='mean'))
    ]:
        estimator.fit(train[features], train[target])
        pred = estimator.predict(test[features])
        rows.append({
            'Fold': fold, 'Model': label,
            'Train_N': len(train), 'Test_N': len(test),
            'Last_Train_Date': train['Date'].max().date().isoformat(),
            'First_Test_Date': test['Date'].min().date().isoformat(),
            'MAE': mean_absolute_error(test[target], pred),
            'RMSE': np.sqrt(mean_squared_error(test[target], pred)),
            'R2': r2_score(test[target], pred)
        })

corrected_cv = pd.DataFrame(rows)
print('Date-grouped chronological cross-validation:')
display(corrected_cv.round(3))
print('Average performance by model:')
display(corrected_cv.groupby('Model')[['MAE', 'RMSE', 'R2']].mean().round(3))
